# 05. Runnable 기본 조합 (LCEL)

- LangChain에서의 Runnable은 입력값을 받아 특정 작업을 수행하고 출력값을 반환하는 실행 가능한 구성요소임.
- 이를 활용하면 프롬프트, 모델, 출력 파서, 사용자 정의 함수, 병렬 처리 흐름을 직관적인 파이프라인으로 표현할 수 있음.
- LCEL(LangChain Expression Language)은 `|` 연산자로 컴포넌트를 체인처럼 연결함.(중요)
- Runnable 조합을 사람이 읽기 쉬운 파이프라인 문법으로 표현한 것.


```
prompt | llm | parser
```

In [3]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
load_dotenv(override=True, dotenv_path="../.env")

True

In [4]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-5.5",
    temperature=0,
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
    use_responses_api=False,
)

## 1. 기본 체인: prompt | llm | parser

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}에 대해 한 문장으로 설명해줘")
])

parser = StrOutputParser()

# | (파이프라인) 연산자로 체인 구성
chain = prompt | llm | parser

result = chain.invoke({"topic": "벡터 데이터베이스"})
print(result) # 벡터 DB의 두가지 역할

벡터 데이터베이스는 텍스트·이미지·음성 같은 데이터를 벡터로 변환해 저장하고, 의미적으로 유사한 데이터를 빠르게 검색할 수 있게 해주는 데이터베이스입니다.


## 2. 체인 스트리밍
- LangChain 체인 실행 결과를 스트리밍 방식으로 조금씩 받아 출력하는 코드

In [7]:
for chunk in chain.stream({"topic": "임베딩"}):
    # 현재 받은 응답 조각을 출력 후 줄바꿈 없이, 버퍼에 쌓아두지 않고 즉시 출력
    print(chunk, end="", flush=True)

임베딩은 텍스트·이미지 같은 데이터를 컴퓨터가 의미적으로 비교하고 처리할 수 있도록 숫자 벡터로 변환한 표현입니다.

In [8]:
for chunk in chain.stream({"topic": "임베딩"}):
    # Python이 출력 내용을 잠시 버퍼에 모아두었다가 나중에 보여줌
    print(chunk, end="", flush=False)

임베딩은 단어·문장·이미지 같은 데이터를 컴퓨터가 의미적 유사성을 계산할 수 있도록 숫자 벡터로 변환한 표현입니다.

# 3. Runnable의 공통 실행 메서드
- 입력 값을 받아 어떤 작업을 수행하고, 출력값을 반환하는 표준 실행 단위
- 프롬프트, LLM, 출력파서, Retriever, 사용자 정의 함수 등을 하나의 실행 규칙으로 연결하기 위한 표준 인터페이스
- 기본 : 입력 → Runnable → 출력
- 여러 Runnable 연결 : 입력 → prompt → llm → parser → custom function → 최종 출력 

- Runnable의 공통 실행 메서드 

| 메서드                    | 의미              | 사용 상황           |
| ---------------------- | --------------- | --------------- |
| `invoke()`             | 단일 입력 실행        | 가장 기본 실행        |
| `ainvoke()`            | 비동기 단일 실행       | async 환경        |
| `batch()`              | 여러 입력을 한 번에 실행  | 여러 질문 일괄 처리     |
| `abatch()`             | 비동기 batch 실행    | 대량 처리           |
| `stream()`             | 결과를 조각 단위로 스트리밍 | 챗봇 실시간 출력       |
| `astream()`            | 비동기 스트리밍        | 웹소켓, SSE 등      |
| `batch_as_completed()` | 완료되는 순서대로 결과 반환 | 입력별 처리 시간이 다를 때 |


| 클래스 | 역할 | 입력 처리 | 결과 |
|---|---|---|---|
| `RunnablePassthrough` | 그대로 전달 | 변경 없음 | 입력과 동일 |
| `RunnableLambda` | 함수 실행 | 원하는 방식으로 변환 | 함수 반환값 |
| `RunnableParallel` | 여러 작업 실행 | 같은 입력을 여러 Runnable에 전달 | dict 형태 |

## 3. RunnablePassthrough - 입력을 그대로 전달
- RunnablePassthrough는 입력값을 거의 그대로 통과시킴

In [9]:
from langchain_core.runnables import RunnablePassthrough

# 입력값을 그대로 흘려보내기
chain = RunnablePassthrough()
result = chain.invoke({"topic": "RAG"})
print(result)

{'topic': 'RAG'}


- `.assign()`은 기존 딕셔너리에 새로운 필드를 추가할 때 사용함

In [10]:
# 실용 예: 입력을 그대로 유지하면서 추가 키 삽입
from langchain_core.runnables import RunnablePassthrough

upper=lambda x: x["word"].upper()
chain = RunnablePassthrough.assign(upper=upper)

result = chain.invoke({"word": "hello"})
print(result)  # {'word': 'hello', 'upper': 'HELLO'}

{'word': 'hello', 'upper': 'HELLO'}


① 기본: 입력을 그대로 통과

```mermaid
flowchart LR
    A["입력<br/>{'topic': 'RAG'}"] --> B[["RunnablePassthrough()"]]
    B --> C["출력<br/>{'topic': 'RAG'}"]
```

② .assign(): 입력은 유지하고 새 키 추가

```mermaid
flowchart LR
    A["입력<br/>{'word': 'hello'}"] --> P[["RunnablePassthrough.assign(upper=upper)"]]
    P -->|"word 그대로 통과"| M(("합치기"))
    P -->|"upper(x) 실행<br/>x['word'].upper()"| M
    M --> C["출력<br/>{'word': 'hello', 'upper': 'HELLO'}"]
```

③ 실제로 쓰는 곳: RAG 체인에서 질문을 그대로 넘기기

```mermaid
flowchart LR
    Q["입력<br/>'RAG가 뭐야?'"] --> R[["retriever"]]
    Q --> P[["RunnablePassthrough()"]]
    R -->|"관련 문서 검색"| D["{'context': 문서들,<br/>'question': 'RAG가 뭐야?'}"]
    P -->|"질문 그대로"| D
    D --> PR[["prompt"]] --> L[["llm"]] --> O[["parser"]] --> F["최종 답변"]
```

```mermaid
flowchart LR
    IN["입력 x"] --> PT[["RunnablePassthrough"]]
    PT -->|"아무것도 안 함"| OUT1["출력 x"]
    PT -->|".assign(key=함수)"| OUT2["출력 x + {key: 함수(x)}"]
```

```python
chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm | parser
```

## 4. RunnableLambda - 함수를 Runnable로 감싸기
- RunnableLambda는 일반 파이썬 함수를 Runnable로 감싸는 객체
- RunnableLambda는 체인 중간이나 마지막에 후처리 로직을 넣을 때 주로 사용

In [11]:
from langchain_core.runnables import RunnableLambda

# 문자열 -> 문자열 편집 반환 함수
def add_prefix(text: str) -> str:
    return f"[결과] {text}"

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}이 뭔가요?")
])

chain = prompt | llm | StrOutputParser() | RunnableLambda(add_prefix)

result = chain.invoke({"topic": "LangGraph"})
print(result)

[결과] LangGraph는 **LLM 애플리케이션이나 AI 에이전트의 흐름을 그래프 형태로 설계하고 실행할 수 있게 해주는 프레임워크**입니다.  
LangChain 생태계의 일부로, 복잡한 에이전트 워크플로우를 만들 때 많이 사용됩니다.

쉽게 말하면:

> “LLM이 어떤 순서로 생각하고, 도구를 쓰고, 결과를 검토하고, 다시 시도할지”를  
> **노드와 엣지로 구성하는 도구**입니다.

---

## 핵심 개념

LangGraph에서는 작업 흐름을 다음처럼 표현합니다.

```text
사용자 입력
   ↓
LLM 호출
   ↓
도구 사용 여부 판단
   ↓          ↘
도구 실행      최종 답변
   ↓
다시 LLM 호출
```

이런 흐름을 코드에서 **그래프**로 정의합니다.

---

## 주요 구성 요소

### 1. State

워크플로우 전체에서 공유되는 상태입니다.

예를 들어:

```python
{
  "messages": [...],
  "user_id": "123",
  "search_results": [...]
}
```

LLM 응답, 도구 실행 결과, 사용자 입력 등을 상태에 저장하면서 흐름을 이어갑니다.

---

### 2. Node

각 작업 단계를 의미합니다.

예를 들어:

- LLM 호출
- 검색 도구 실행
- 데이터베이스 조회
- 사용자 승인 요청
- 결과 검증

---

### 3. Edge

노드 간의 연결입니다.

예를 들어:

```text
LLM 노드 → 도구 실행 노드
도구 실행 노드 → LLM 노드
LLM 노드 → 종료
```

조건에 따라 다른 노드로 이동하는 **조건부 분기**도 만들 수 있습니다.

---

## LangGraph가 필요한 이유

일반적인 LangChain 체인은 보통 다음처럼 직선적인 흐름에 강합니다.

```text
입력 → 프롬프트 → LLM → 출력
```

하지만 실제 에이전트는 더 복잡합니다.

예를 들어:

```text
사용자 질문
→ LLM이 검색 필요 

## 5. 병렬 실행 - RunnableParallel

딕셔너리 형태로 여러 체인을 동시에 실행합니다.

In [14]:
from langchain_core.runnables import RunnableParallel

prompt_ko = ChatPromptTemplate.from_messages([("human", "{topic}을 한국어로 설명해줘")])
prompt_en = ChatPromptTemplate.from_messages([("human", "Explain {topic} in English")])

chain_ko = prompt_ko | llm | StrOutputParser()
chain_en = prompt_en | llm | StrOutputParser()

# 두 체인을 병렬로 실행하여 각각 한국어와 영어 설명 생성
parallel_chain = RunnableParallel(
    korean=chain_ko,
    english=chain_en
)

result = parallel_chain.invoke({"topic": "RAG"})
print("[한국어]", result["korean"])
print("[English]", result["english"])

[한국어] RAG는 "Retrieval-Augmented Generation"의 약자로, 정보 검색과 생성 모델을 결합한 접근 방식을 의미합니다. 이 방법은 주로 자연어 처리(NLP) 분야에서 사용되며, 특히 질문 응답 시스템이나 대화형 AI 모델에서 효과적입니다.

RAG의 기본 원리는 다음과 같습니다:

1. **정보 검색**: 사용자가 질문을 입력하면, 시스템은 관련된 정보를 데이터베이스나 문서에서 검색합니다. 이 단계에서는 검색 엔진이나 데이터베이스 쿼리를 사용하여 질문과 관련된 문서나 정보를 찾아냅니다.

2. **정보 생성**: 검색된 정보를 바탕으로, 생성 모델이 자연어로 응답을 생성합니다. 이 과정에서는 검색된 내용을 요약하거나, 질문에 대한 구체적인 답변을 작성합니다.

이러한 방식의 장점은 모델이 사전에 학습한 지식뿐만 아니라, 최신 정보나 특정 도메인에 대한 세부 정보를 활용할 수 있다는 점입니다. 따라서 RAG는 더 정확하고 유용한 응답을 제공할 수 있습니다. 

RAG는 특히 대규모 데이터셋을 활용할 수 있는 환경에서 강력한 성능을 발휘하며, 정보의 정확성과 관련성을 높이는 데 기여합니다.
[English] RAG stands for "Red, Amber, Green," which is a color-coding system used to indicate the status or performance of a project, task, or situation. Each color represents a different level of urgency or concern:

- **Red**: This indicates a serious issue or problem that requires immediate attention. It suggests that the project is off track, facing significant challenges, or at risk of failing to meet its objectives.

## 실습 문제
Q2. RunnableLambda를 활용해 LLM 응답을 받아 단어 수를 세는 함수를 파이프라인 끝에 추가해보세요. 결과로 {'text': '...', 'word_count': 42} 형태로 반환하세요.

In [ ]:
from langchain_core.runnables import RunnableLambda

# 1. 프롬프트 구성
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 초보자도 이해할 수 있게 설명해줘."
)

# 2. LLM 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 3. 출력 파서 생성
parser = StrOutputParser()

# 4. LLM 응답을 받아 단어 수를 세는 함수
def count_words(text: str) -> dict:
    return {
        "text": text,
        "word_count": len(text.split())
    }

# 5. RunnableLambda로 함수 감싸기
word_count_runnable = RunnableLambda(count_words)

# 6. LCEL 체인 구성
chain = prompt | llm | parser | word_count_runnable

# 7. 실행
result = chain.invoke({"topic": "LangChain Runnable"})

print(result["word_count"])  # 설명의 단어 수
print("-"*20)
print(result["text"])        # LLM이 생성한 설명

175
--------------------
LangChain은 자연어 처리(NLP)와 관련된 다양한 작업을 수행할 수 있도록 도와주는 프레임워크입니다. 그 중에서도 "Runnable"은 LangChain에서 작업을 수행하는 기본 단위입니다. 초보자도 이해할 수 있도록 쉽게 설명해볼게요.

### Runnable이란?

1. **작업 단위**: Runnable은 특정 작업을 수행하는 코드 블록입니다. 예를 들어, 텍스트를 분석하거나, 질문에 답변하거나, 데이터를 변환하는 등의 작업을 할 수 있습니다.

2. **입력과 출력**: Runnable은 입력을 받아서 처리한 후, 결과를 출력합니다. 예를 들어, 사용자가 "안녕하세요"라는 텍스트를 입력하면, Runnable은 이를 처리하여 "안녕하세요! 어떻게 도와드릴까요?"와 같은 응답을 생성할 수 있습니다.

3. **조합 가능성**: 여러 개의 Runnable을 조합하여 더 복잡한 작업을 수행할 수 있습니다. 예를 들어, 첫 번째 Runnable이 텍스트를 정리하고, 두 번째 Runnable이 그 텍스트를 분석하는 식으로 연결할 수 있습니다.

### 예시

- **텍스트 변환**: 사용자가 입력한 문장을 대문자로 변환하는 Runnable을 만들 수 있습니다.
- **질문 응답**: 사용자가 질문을 입력하면, 그에 대한 답변을 생성하는 Runnable을 만들 수 있습니다.

### 요약

Runnable은 LangChain에서 작업을 수행하는 기본적인 구성 요소로, 입력을 받아서 처리하고 출력을 생성하는 역할을 합니다. 여러 개의 Runnable을 조합하여 복잡한 작업을 수행할 수 있는 유연성을 제공합니다. 초보자도 쉽게 사용할 수 있도록 설계되어 있어, 자연어 처리 작업을 간편하게 수행할 수 있습니다.
